In [ ]:
'''
Explanation
Data Loading and Summary: The main function starts by loading the CSV file and providing a summary of the data (e.g., number of clients, average loan amounts by gender).

Visualizations: It uses plot_graph for histogram and scatter plot visualizations, showing interest rates by gender and their distribution across credit scores.

Credit Score and Interest Rate Bins: Defines bins for CreditScore and InterestRate to facilitate grouped analysis.

Calculating Metrics:

Weighted Disparity Index (WDI): Calculates and displays the WDI to quantify interest rate differences between genders.
Equality of Opportunity (EOP): Calculates and visualizes EOP metrics across credit score bins.
Fairness Cost Calculations:

Fixed Discount: Calculates the cost for applying a fixed discount to interest rate differences up to a specific credit score bin.
Varying Fairness Degrees: Iterates over different fairness degrees and credit score bins, plotting the associated costs.
Decaying Discount: Evaluates cost using a decaying discount model to gradually decrease fairness adjustments as credit scores decrease.
'''

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.VisualizationPKG import plot_graph
from src.DiscriminationMetrics import calculate_weighted_disparity_index, calculate_equality_of_opportunity, equality_of_opportunity_M1
from src.CostOfFairness import calculate_fairness_cost, calculate_fairness_cost_with_decay

def main(file_path):
    print()
    # Load dataset
    df = pd.read_csv(file_path)
    # Printing the general statistics
    print('Number of clients =', len(df))
    print('Number of male clients =', len(df[df.Gender == 'Male']))
    print('Number of female clients =', len(df[df.Gender == 'Female']))
    print('Average loan amount =', round(df.LoanAmount.mean(), 2))
    print('Average loan amount for women =', round(df[df.Gender == 'Female'].LoanAmount.mean(), 2))
    print('Average loan amount for men =', round(df[df.Gender == 'Male'].LoanAmount.mean(), 2))
    print('Number of clients with no default =', len(df[df.Default == 0]))
    print('Average Interest rate =', round(df.InterestRate.mean(), 2))
    print('Average Interest rate for men =', round(df[df.Gender == 'Male'].InterestRate.mean(), 2))
    print('Average Interest rate for women =', round(df[df.Gender == 'Female'].InterestRate.mean(), 2))

    # Visualizations
    plot_graph(df, 'InterestRate', '', 'histogram', 'Gender', ['Male', 'Female'], '', 'Interest Rate', 'Density', 'Histogram of Interest Rate')
    plot_graph(df, 'CreditScore', 'InterestRate', 'scatter', 'Gender', ['Male', 'Female'], '', 'Credit Score', 'Interest Rate', 'Interest Rate by Credit Score')

    # Define Credit Score bins
    cs_bins = [0, 0.03, 0.04, 0.05, 0.08, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.9]
    
    cs_labels = [f'Cat {i + 1}' for i in range(len(cs_bins) - 1)]
    
    df=df.sort_values('CreditScore')
    df['CreditScoreBin'] = pd.cut(df['CreditScore'], bins=cs_bins, labels=cs_labels, include_lowest=True)

    # Define Interest Rate bins
    ir_bins = [80, 85, 90, 95, 100, 105, 110, 120, 130, 150, 160, 170]
    ir_labels = [f'{ir_bins[i]}-{ir_bins[i + 1]}' for i in range(len(ir_bins) - 1)]
    df['InterestRateBin'] = pd.cut(df['InterestRate'], bins=ir_bins, labels=ir_labels, include_lowest=True)

    # Calculate Weighted Disparity Index (WDI)
    bin_ir_summary = df.groupby(['InterestRateBin', 'Gender'], observed=False).size().unstack(fill_value=0)
    wdi = calculate_weighted_disparity_index(bin_ir_summary, 'Female', 'Male')
    print(f"Weighted Disparity Index (WDI): {wdi:.2f}")

    # Calculate Equality of Opportunity
    eop, eop_bin, bins_eop = calculate_equality_of_opportunity(df, 'Gender', 'Female', 'Male', 'InterestRate', 'CreditScoreBin')
    print('Equality of Opportunity Index =', eop)
    print('eop bin', eop_bin)
    print('bins', bins_eop)
    
    df_eop = pd.DataFrame({'bins': df['CreditScoreBin'].unique(), 'eop': eop_bin})

    # printing the inequality in each credit score bin
    plot_graph(df_eop, 'bins', 'eop', 'solo_bins', '', '', '', 'Credit Score Bins', 'Equality of Opportunity', 'Equality of Opportunity by Credit Score Bins')


    # Calculate fairness cost with a fixed discount
    fairness_cost = calculate_fairness_cost(df, 'LoanAmount', 'CreditScoreBin', 'Gender', 'Male', 'Female', 'InterestRate', 0.2, 4, 'Default')
    print(f"The cost for the bank to reduce the rate difference by 20% in the first 4 credit score bins is: ${round(fairness_cost / 1e6, 2)} million")

    # Varying the fairness degree and max category for fairness cost
    fairness_degrees = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]
    max_category = df['CreditScoreBin'].nunique()
    fairness_cost_matrix = {}

    for max_cat in range(1, max_category + 1):
        costs = [calculate_fairness_cost(df, 'LoanAmount', 'CreditScoreBin', 'Gender', 'Male', 'Female', 'InterestRate', degree, max_cat, 'Default') for degree in fairness_degrees]
        fairness_cost_matrix[f'Cat {max_cat}'] = costs
        plt.plot(fairness_degrees, costs, label=f'Cat {max_cat}')

    plt.xlabel('Interest Rate Fairness Discount')
    plt.ylabel('Cost in Millions of $')
    plt.title("Cost to Adjust Female Interest Rate to Match Male Rates")
    plt.legend(title='Max Credit Score Bin')
    plt.show()

    fairness_cost_df = pd.DataFrame(fairness_cost_matrix, index=fairness_degrees)
    print(fairness_cost_df)

    # Calculate fairness cost with decaying discount
    max_discount = 0.5
    min_discount = 0.1
    decay_factor = 0.1
    decaying_cost = calculate_fairness_cost_with_decay(df, 'LoanAmount', 'CreditScoreBin', 'Gender', 'InterestRate', 'Female', 'Male', max_discount, min_discount, 2, decay_factor)
    print(f"The cost for the bank to apply a decaying discount is: ${round(decaying_cost / 1e6, 2)} million")

# Run main function with file path
file_path = 'Simulated_Dataset.csv'
main(file_path)
